In [1]:
# Your Name, Roll Number, 2026-11-23
import sys, os, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import pandas as pd, numpy as np, json
import matplotlib.pyplot as plt

errors = pd.read_csv("outputs/reports/errors.csv")
comparison = pd.read_csv("outputs/reports/model_comparison.csv")
metrics = json.loads((pathlib.Path("outputs/reports/test_metrics.json")).read_text())
invoice = pd.read_csv("outputs/reports/invoice_level_dataset.csv")

print("Errors:", len(errors), "of", len(invoice) // 5, "test rows approx")
print("Test metrics:", metrics)
comparison

Errors: 204 of 3365 test rows approx
Test metrics: {'model': 'gb', 'precision': 0.9322033898305084, 'recall': 0.9477124183006536, 'f1': 0.9398939304655274, 'roc_auc': 0.9874993336249496, 'pr_auc': 0.9869336297962832}


,model,precision,recall,f1,roc_auc,pr_auc
0,gb,0.940762,0.947994,0.944339,0.988570,0.987858
1,rf,0.933947,0.936701,0.935315,0.983932,0.982644
2,logreg,0.918711,0.897920,0.908172,0.973575,0.971970
3,dummy,0.000000,0.000000,0.000000,0.500000,0.500000


In [2]:
err = errors.copy()
err["is_error"] = (err["y_true"] != err["y_pred"]).astype(int)

by_country = (
    err.groupby("country")
       .agg(n=("is_error", "size"), err_rate=("is_error", "mean"))
       .query("n >= 10")
       .sort_values("err_rate", ascending=False)
       .head(10)
)
by_country

,n,err_rate
country,,
United Kingdom,187,1.0


In [3]:
err["lines_bucket"] = pd.cut(
    err["n_lines"],
    bins=[0, 1, 3, 10, 100, 10000],
    labels=["1", "2-3", "4-10", "11-100", "100+"],
)
by_lines = err.groupby("lines_bucket", observed=True)["is_error"].agg(["size", "mean"])
print(by_lines)

              size  mean
lines_bucket            
2-3              3   1.0
4-10            63   1.0
11-100         138   1.0


In [4]:
by_hour = err.groupby("invoice_hour")["is_error"].agg(["size", "mean"])
by_hour

,size,mean
invoice_hour,,
8,5,1.0
9,8,1.0
10,20,1.0
11,31,1.0
12,29,1.0
13,36,1.0
14,24,1.0
15,26,1.0
16,14,1.0


In [5]:
os.makedirs("outputs/figures", exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))
by_lines["mean"].plot(kind="bar", ax=ax, color="indianred")
ax.set_title("Error rate by number of distinct products per invoice")
ax.set_xlabel("Distinct products per invoice (bucket)")
ax.set_ylabel("Error rate (proportion misclassified)")
plt.tight_layout()
fig.savefig("outputs/figures/error_by_basket_size.png", dpi=150)
plt.close(fig)

print("Saved error_by_basket_size.png")


Saved error_by_basket_size.png


In [6]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(err.loc[err.y_true == 0, "y_proba"], bins=30, alpha=0.6,
        label="True=0", color="steelblue")
ax.hist(err.loc[err.y_true == 1, "y_proba"], bins=30, alpha=0.6,
        label="True=1", color="darkorange")
ax.set_title("Predicted probability of 'high value' by true class (test set)")
ax.set_xlabel("Predicted probability")
ax.set_ylabel("Number of rows (count)")
ax.legend()
plt.tight_layout()
fig.savefig("outputs/figures/proba_distribution.png", dpi=150)
plt.close(fig)

print("Saved proba_distribution.png")

Saved proba_distribution.png


In [7]:
fn = err[(err.y_true == 1) & (err.y_pred == 0)].head(3)
fp = err[(err.y_true == 0) & (err.y_pred == 1)].head(3)

print("=== FALSE NEGATIVES (missed high-value orders) ===")
print(fn[["n_lines", "total_quantity", "avg_unit_price", "max_unit_price",
          "country", "y_proba"]].to_string())

print("\n=== FALSE POSITIVES (falsely flagged high-value) ===")
print(fp[["n_lines", "total_quantity", "avg_unit_price", "max_unit_price",
          "country", "y_proba"]].to_string())

=== FALSE NEGATIVES (missed high-value orders) ===
   n_lines  total_quantity  avg_unit_price  max_unit_price         country   y_proba
1       33             154        1.246757            2.95  United Kingdom  0.184542
4        3              39        5.950000            5.95  United Kingdom  0.443457
6        5              78        2.256000            3.75  United Kingdom  0.057525

=== FALSE POSITIVES (falsely flagged high-value) ===
   n_lines  total_quantity  avg_unit_price  max_unit_price         country   y_proba
0       44             155        1.666818            4.15  United Kingdom  0.740275
2       47             231        1.100833            5.75  United Kingdom  0.558172
3       13              95        2.678462            6.25  United Kingdom  0.573448


In [8]:
from sklearn.inspection import permutation_importance
import joblib

pipe = joblib.load("models/pipeline.joblib")
X_test = err[["n_lines", "total_quantity", "avg_unit_price", "max_unit_price",
              "n_countries", "invoice_month", "invoice_dow", "invoice_hour",
              "country"]]
y_test = err["y_true"]

r = permutation_importance(pipe, X_test, y_test, n_repeats=10,
                            random_state=42, scoring="f1")
imp = pd.Series(r.importances_mean, index=X_test.columns).sort_values(ascending=False)
print("Top features by permutation importance (F1):")
print(imp.head(10))
imp.to_csv("outputs/reports/permutation_importance.csv")

Top features by permutation importance (F1):
n_countries       0.000000
country           0.000000
invoice_month    -0.002927
invoice_dow      -0.024254
invoice_hour     -0.041356
n_lines          -0.081869
max_unit_price   -0.118298
avg_unit_price   -0.409454
total_quantity   -0.454189
dtype: float64
